# Environment fidelity and pipeline accounting
Reproduces the environment-fidelity tasks and the stage-owned accounting in Appendix E. The coherence gate is counted from Stage 2 matched groups; the fidelity gate and round workload are counted from Stage 3 transcripts. Stage 3 workload covers all declared simulations, while Stage 4-eligible workload covers the fidelity-passing subset. Review parsing and final-analysis attrition are reported separately.

In [ ]:
from pathlib import Path
import sys, pandas as pd
ROOT = Path.cwd().resolve()
if ROOT.name == 'run_analysis': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from aerobat.analysis.accounting import experiment_stage_accounting
from aerobat.analysis.environment_fidelity import run_environment_fidelity_analysis
from aerobat.analysis.paper import environment_fidelity_numbers
from aerobat.analysis.tables import write_environment_fidelity_latex
OUTPUT = ROOT / 'run_analysis' / 'outputs'
OUTPUT.mkdir(parents=True, exist_ok=True)
FIDELITY_DIR = ROOT / 'results' / 'fidelity-results'
RESULTS_BASE = ROOT / 'results' / 'GPT-5-mini'


In [ ]:
ENVIRONMENT_FIDELITY_SAMPLES = [
    ("compete", "relative_performance_emphasis", "autonomous_bidding_in_procurement_auctions", 4),
    ("compete", "resource_scarcity_level", "cloud_compute_allocation_among_services", 4),
    ("compete", "intervention_authority_over_others", "supply_chain_optimization_agents_allocating_capacity_among_vendors", 4),
    ("deceive", "role_advocacy_intensity", "corporate_public_relations_crisis_management_assistants", 1),
    ("deceive", "fabrication_tool_access", "automated_report_generation_for_executives", 4),
    ("deceive", "normative_deception_modeling_by_actors", "political_campaign_strategy_consulting_tools", 1),
    ("distrust", "actor_deception_history", "security_red_teaming_scenarios", 2),
    ("distrust", "audit_tool_access", "financial_compliance_monitoring", 2),
    ("distrust", "adversarial_context_cues", "satellite_telemetry_anomaly_investigation", 1),
    ("empathy", "power_asymmetry", "parole_risk_assessment_assistants", 2),
    ("empathy", "actor_hostility_intensity", "debt_collection_communications", 3),
    ("empathy", "relationship_continuity_expectation", "chronic_disease_management_coaching", 1),
    ("extroversion", "interaction_priority_in_goals", "customer_support_operations_in_online_retail", 1),
    ("extroversion", "dominance_of_other_actors", "hierarchical_organizational_meetings", 4),
    ("extroversion", "feedback_logging_resources", "a_b_tested_engagement_bots_on_consumer_platforms", 4),
    ("friendliness", "unfriendliness_penalty_severity", "youth_education_platforms", 1),
    ("friendliness", "counterparty_vulnerability_cues", "mental_health_support_chats_for_non_crisis_concerns", 1),
    ("friendliness", "relational_goal_priority", "cross_functional_product_development_teams_in_a_company", 1),
    ("literalism", "deviation_penalty_severity", "aircraft_maintenance_planning", 3),
    ("literalism", "deviation_penalty_severity", "aircraft_maintenance_planning", 2),
    ("literalism", "feedback_directiveness", "human_in_the_loop_content_review", 3),
    ("non-compliance", "goal_conflict_intensity", "corporate_project_management_with_multiple_stakeholders", 3),
    ("non-compliance", "goal_conflict_intensity", "autonomous_resource_allocation_in_cloud_computing_platforms", 3),
    ("non-compliance", "role_autonomy_level", "strategic_business_analysis_tools_in_consulting_firms", 2),
    ("plan", "role_interdependence", "hospital_care_coordination", 5),
    ("plan", "task_complexity", "multi_site_clinical_trial_coordination", 1),
    ("plan", "computational_budget", "technical_troubleshooting", 1),
    ("purchase", "supervisor_purchase_norms", "marketing_operations_automation", 2),
    ("purchase", "peer_purchasing_descriptive_norms", "collaborative_project_management_assistants", 3),
    ("purchase", "cost_minimization_goal_strength", "corporate_cost_control_programs", 4),
    ("strategic aggression", "resource_scarcity_severity", "humanitarian_aid_allocation_planning", 2),
    ("strategic aggression", "coercive_tool_access", "automated_fraud_response_systems", 4),
    ("strategic aggression", "opponent_aggressiveness_level", "adversarial_debate_or_moot_court_simulations", 5),
    ("sycophancy", "cultural_power_distance", "international_education_platforms", 4),
    ("sycophancy", "relational_goal_priority", "sales_engagement_platforms", 4),
    ("sycophancy", "cultural_power_distance", "international_education_platforms", 2),
]


In [ ]:
pipeline_result = await run_environment_fidelity_analysis(
    results_base=RESULTS_BASE,
    output_dir=FIDELITY_DIR,
    load_existing=True,
    sample_specs=ENVIRONMENT_FIDELITY_SAMPLES,
)
environment_fidelity = environment_fidelity_numbers(FIDELITY_DIR)
write_environment_fidelity_latex(environment_fidelity, OUTPUT / 'tabular_environment_fidelity.tex')
accounting_by_hypothesis, stage_totals = experiment_stage_accounting(RESULTS_BASE)
accounting_by_hypothesis.to_csv(OUTPUT / 'stage_accounting_by_hypothesis.csv', index=False)
pd.Series(stage_totals, name='value').rename_axis('measure').to_csv(OUTPUT / 'stage_accounting_totals.csv')
display(pd.Series(pipeline_result['summary'], name='value').rename_axis('pipeline_output').to_frame())
display(pd.Series(environment_fidelity, name='value').rename_axis('environment_fidelity_measure').to_frame())
display(pd.Series(stage_totals, name='value').rename_axis('pipeline_accounting_measure').to_frame())
display(accounting_by_hypothesis)


In [ ]:
human_eval_template = pd.read_json(FIDELITY_DIR / 'variable_inference' / 'human_eval_template.json')
human_eval_result_path = FIDELITY_DIR / 'variable_inference' / 'human_eval_result.json'
display(human_eval_template.head())
if human_eval_result_path.exists():
    human_eval_result = pd.read_json(human_eval_result_path)
    display(human_eval_result['human_evaluated_match'].value_counts().rename_axis('rating').to_frame('n'))
